In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from tricor.differentiable_pdf import DifferentiablePDFADF, DifferentiableSpectralLoss
from tricor.pdf_adf_widget import PDFADFWidget

## Build test structures

Silicon diamond and NaCl supercells for testing. These are created directly as torch tensors (no ASE dependency needed).

In [ ]:
def make_si_diamond(repeat=(3, 3, 3)):
    """Create a silicon diamond supercell as torch tensors."""
    a = 5.43
    basis_frac = np.array([
        [0.00, 0.00, 0.00],
        [0.50, 0.50, 0.00],
        [0.50, 0.00, 0.50],
        [0.00, 0.50, 0.50],
        [0.25, 0.25, 0.25],
        [0.75, 0.75, 0.25],
        [0.75, 0.25, 0.75],
        [0.25, 0.75, 0.75],
    ])
    nx, ny, nz = repeat
    positions = []
    for ix in range(nx):
        for iy in range(ny):
            for iz in range(nz):
                for b in basis_frac:
                    positions.append((b + [ix, iy, iz]) * a)
    positions = np.array(positions)
    cell = np.diag([a * nx, a * ny, a * nz])
    n_atoms = len(positions)

    positions_t = torch.tensor(positions, dtype=torch.float64, requires_grad=True)
    species_t = torch.full((n_atoms,), 14, dtype=torch.long)
    cell_t = torch.tensor(cell, dtype=torch.float64)
    return positions_t, species_t, cell_t


def make_nacl(repeat=(2, 2, 2)):
    """Create a NaCl rocksalt supercell as torch tensors."""
    a = 5.64
    fcc = np.array([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]])

    nx, ny, nz = repeat
    positions, species_list = [], []
    for ix in range(nx):
        for iy in range(ny):
            for iz in range(nz):
                offset = np.array([ix, iy, iz])
                for b in fcc:
                    positions.append((b + offset) * a)
                    species_list.append(11)  # Na
                for b in fcc + 0.5:
                    positions.append((b + offset) * a)
                    species_list.append(17)  # Cl
    positions = np.array(positions)
    cell = np.diag([a * nx, a * ny, a * nz])

    positions_t = torch.tensor(positions, dtype=torch.float64, requires_grad=True)
    species_t = torch.tensor(species_list, dtype=torch.long)
    cell_t = torch.tensor(cell, dtype=torch.float64)
    return positions_t, species_t, cell_t

In [ ]:
positions, species, cell = make_si_diamond((3, 3, 3))
print(f"Si diamond: {positions.shape[0]} atoms, cell = {cell.diag().numpy()} A")

## Compute g2(r) and ADF for Si diamond

Single-species system: 1 pair type (Si-Si), 1 triplet type (Si-Si-Si). Using r_max=8 A to capture several neighbor shells.

In [ ]:
calc_si = DifferentiablePDFADF(
    r_max=8.0,
    r_step=0.05,
    phi_num_bins=90,
    sigma_r=0.1,
    sigma_phi=0.05,
    species=[14],
).double()

g2_si, adf_si = calc_si.compute(positions, species, cell)
print(f"g2 shape:  {g2_si.shape}  (num_species, num_species, num_r)")
print(f"ADF shape: {adf_si.shape}  (num_triplets, phi_num_bins)")
print(f"Pair labels:    {calc_si.pair_labels}")
print(f"Triplet labels: {calc_si.triplet_labels}")

## Plot g2(r) and ADF with matplotlib

In [ ]:
r = calc_si.r_grid.numpy()
phi_deg = np.rad2deg(calc_si.phi_grid.numpy())

g2_np = g2_si[0, 0].detach().numpy()
adf_np = adf_si[0].detach().numpy()

# normalize g2 by r^2 to get standard g(r) shape
g2_norm = g2_np / np.maximum(r**2, 1e-12)
tail = g2_norm[int(0.7 * len(r)):]
g2_norm /= np.mean(tail[tail > 0]) if np.any(tail > 0) else 1.0

# normalize ADF to probability density
dphi = np.deg2rad(phi_deg[1] - phi_deg[0]) if len(phi_deg) > 1 else 1.0
adf_norm = adf_np / (adf_np.sum() * dphi) if adf_np.sum() > 0 else adf_np

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(r, g2_norm, lw=2, color="#1b6370")
axs[0].axhline(1.0, color="k", ls="--", lw=1, label="g(r)=1")
axs[0].set_xlabel("r (A)")
axs[0].set_ylabel("g(r)")
axs[0].set_title("Si-Si pair distribution function")
axs[0].legend()

axs[1].fill_between(phi_deg, adf_norm, alpha=0.15, color="#6b3a7d")
axs[1].plot(phi_deg, adf_norm, lw=2, color="#6b3a7d")
axs[1].axvline(109.47, color="k", ls="--", lw=1, label="tetrahedral 109.47$\\degree$")
axs[1].set_xlabel("angle (deg)")
axs[1].set_ylabel("P(phi)")
axs[1].set_title("Si-Si-Si angular distribution")
axs[1].legend()

plt.tight_layout()
plt.show()

## Interactive widget

In [ ]:
PDFADFWidget(calc_si, g2_si, adf_si)

## Effect of r_max on ADF

With a tight cutoff (r_max=2.6 A), only first neighbors contribute -- the ADF shows a clean tetrahedral peak. With a larger cutoff, second and further neighbor shells add additional angle contributions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for r_cut, color, ls in [(2.6, "#2d6a4f", "-"), (4.0, "#e07a2f", "-"), (6.0, "#7b2d8e", "--"), (8.0, "#1b6370", ":")]:
    calc_tmp = DifferentiablePDFADF(
        r_max=r_cut, r_step=0.1, phi_num_bins=180,
        sigma_r=0.1, sigma_phi=0.04, species=[14],
    ).double()
    _, adf_tmp = calc_tmp.compute(positions, species, cell)
    phi_d = np.rad2deg(calc_tmp.phi_grid.numpy())
    adf_v = adf_tmp[0].detach().numpy()
    dp = np.deg2rad(phi_d[1] - phi_d[0])
    adf_v = adf_v / (adf_v.sum() * dp) if adf_v.sum() > 0 else adf_v
    ax.plot(phi_d, adf_v, lw=2, ls=ls, color=color, label=f"r_max={r_cut} A")

ax.axvline(109.47, color="k", ls="--", lw=1, alpha=0.5)
ax.set_xlabel("angle (deg)")
ax.set_ylabel("P(phi)")
ax.set_title("Si ADF: effect of neighbor cutoff")
ax.legend()
plt.tight_layout()
plt.show()

## Multi-species: NaCl

Two species (Na, Cl) gives 4 pair types for g2 and 6 rooted triplet types for the ADF.

In [ ]:
pos_nacl, sp_nacl, cell_nacl = make_nacl((3, 3, 3))
print(f"NaCl: {pos_nacl.shape[0]} atoms, cell = {cell_nacl.diag().numpy()} A")

calc_nacl = DifferentiablePDFADF(
    r_max=8.0,
    r_step=0.05,
    phi_num_bins=90,
    sigma_r=0.1,
    sigma_phi=0.05,
    species=[11, 17],
).double()

g2_nacl, adf_nacl = calc_nacl.compute(pos_nacl, sp_nacl, cell_nacl)
print(f"g2 shape:  {g2_nacl.shape}")
print(f"ADF shape: {adf_nacl.shape}")
print(f"Pair labels:    {calc_nacl.pair_labels}")
print(f"Triplet labels: {calc_nacl.triplet_labels}")

In [ ]:
r_nacl = calc_nacl.r_grid.numpy()

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# g2 for all 4 pair types
colors = ["#1b6370", "#a05c2c", "#2d6a4f", "#7b2d8e"]
for i in range(calc_nacl.num_species):
    for j in range(calc_nacl.num_species):
        g2_ij = g2_nacl[i, j].detach().numpy()
        g2_ij_norm = g2_ij / np.maximum(r_nacl**2, 1e-12)
        tail = g2_ij_norm[int(0.7 * len(r_nacl)):]
        scale = np.mean(tail[tail > 0]) if np.any(tail > 0) else 1.0
        g2_ij_norm /= scale
        label = calc_nacl.pair_labels[i * calc_nacl.num_species + j]
        axs[0].plot(r_nacl, g2_ij_norm, lw=2, label=label, color=colors[i * calc_nacl.num_species + j])

axs[0].axhline(1.0, color="k", ls="--", lw=1)
axs[0].set_xlabel("r (A)")
axs[0].set_ylabel("g(r)")
axs[0].set_title("NaCl partial PDFs")
axs[0].legend()

# ADF for all 6 triplet types
phi_nacl = np.rad2deg(calc_nacl.phi_grid.numpy())
for tri_idx in range(calc_nacl.num_triplets):
    adf_v = adf_nacl[tri_idx].detach().numpy()
    dp = np.deg2rad(phi_nacl[1] - phi_nacl[0])
    if adf_v.sum() > 0:
        adf_v = adf_v / (adf_v.sum() * dp)
    axs[1].plot(phi_nacl, adf_v, lw=1.5, label=calc_nacl.triplet_labels[tri_idx])

axs[1].set_xlabel("angle (deg)")
axs[1].set_ylabel("P(phi)")
axs[1].set_title("NaCl angular distributions")
axs[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
PDFADFWidget(calc_nacl, g2_nacl, adf_nacl)

## Verify gradient flow

The whole point of this implementation is differentiability. Verify that gradients propagate through both g2 and ADF back to atomic positions.

In [ ]:
pos_test = positions.detach().clone().requires_grad_(True)

g2_test, adf_test = calc_si.compute(pos_test, species, cell)
loss = (g2_test ** 2).sum() + (adf_test ** 2).sum()
loss.backward()

grad = pos_test.grad
print(f"Loss:          {loss.item():.4f}")
print(f"Gradient shape: {grad.shape}")
print(f"Gradient norm:  {grad.norm().item():.6f}")
print(f"Any non-zero:   {not torch.all(grad == 0).item()}")
print(f"All finite:     {torch.isfinite(grad).all().item()}")

## Spectral guidance loss

Demonstrate the GLASS-style guidance: compute a target from the current structure, perturb positions, and show that the guidance vectors point back toward the original structure.

In [ ]:
calc_guide = DifferentiablePDFADF(
    r_max=5.0, r_step=0.1, phi_num_bins=45,
    sigma_r=0.15, sigma_phi=0.1, species=[14],
).double()

loss_fn = DifferentiableSpectralLoss(calc_guide, pdf_weight=1.0, adf_weight=1.0)

# compute target from unperturbed structure
with torch.no_grad():
    target_g2, target_adf = calc_guide.compute(positions, species, cell)

# perturb positions
rng = torch.Generator().manual_seed(42)
perturbation = 0.3 * torch.randn(positions.shape, dtype=torch.float64, generator=rng)
pos_perturbed = (positions.detach() + perturbation).requires_grad_(True)

# compute guidance
guidance, components = loss_fn.compute_guidance(
    pos_perturbed, species, cell, target_g2, target_adf
)

print(f"PDF loss: {components['pdf_loss'].item():.6f}")
print(f"ADF loss: {components['adf_loss'].item():.6f}")
print(f"Guidance shape: {guidance.shape}")
print(f"Guidance norm:  {guidance.norm().item():.4f}")

# check guidance points back toward original positions
displacement = positions.detach() - pos_perturbed.detach()  # vector toward target
cos_sim = torch.nn.functional.cosine_similarity(
    guidance.flatten(), displacement.flatten(), dim=0
)
print(f"\nCosine similarity between guidance and displacement toward target: {cos_sim.item():.4f}")
print("(positive = guidance points toward target structure)")

## Gradient descent on spectral loss

Run a few steps of gradient descent to show the perturbed structure relaxing back toward the target g2/ADF. Track the loss and visualize the g2 before and after.

In [ ]:
# snapshot g2/adf before optimization
with torch.no_grad():
    g2_before, adf_before = calc_guide.compute(pos_perturbed, species, cell)

# gradient descent with normalized steps
pos_opt = pos_perturbed.detach().clone()
step_size = 0.05  # Angstroms per step
losses = []

for step in range(50):
    pos_opt = pos_opt.detach().requires_grad_(True)
    loss, comps = loss_fn(pos_opt, species, cell, target_g2, target_adf)
    losses.append(loss.item())
    grad = torch.autograd.grad(loss, pos_opt)[0]
    # normalize gradient so each step moves atoms by at most step_size
    grad_norm = grad.norm()
    if grad_norm > 1e-12:
        pos_opt = pos_opt - step_size * grad / grad_norm

print(f"Loss: {losses[0]:.2f} -> {losses[-1]:.2f} ({losses[-1]/losses[0]*100:.1f}% of initial)")

# snapshot after
with torch.no_grad():
    g2_after, adf_after = calc_guide.compute(pos_opt, species, cell)

In [ ]:
r_guide = calc_guide.r_grid.numpy()
phi_guide = np.rad2deg(calc_guide.phi_grid.numpy())

fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# loss curve
axs[0].semilogy(losses, lw=2, color="#1b6370")
axs[0].set_xlabel("step")
axs[0].set_ylabel("spectral loss")
axs[0].set_title("Optimization convergence")

# g2 comparison
for data, label, color, ls in [
    (target_g2, "target", "k", "--"),
    (g2_before, "perturbed", "#cf4040", "-"),
    (g2_after, "optimized", "#2d6a4f", "-"),
]:
    g2_v = data[0, 0].detach().numpy()
    g2_v = g2_v / np.maximum(r_guide**2, 1e-12)
    tail = g2_v[int(0.7 * len(r_guide)):]
    g2_v /= np.mean(tail[tail > 0]) if np.any(tail > 0) else 1.0
    axs[1].plot(r_guide, g2_v, lw=2, ls=ls, color=color, label=label)

axs[1].axhline(1.0, color="gray", ls=":", lw=1)
axs[1].set_xlabel("r (A)")
axs[1].set_ylabel("g(r)")
axs[1].set_title("g2 recovery")
axs[1].legend()

# ADF comparison
dphi = np.deg2rad(phi_guide[1] - phi_guide[0])
for data, label, color, ls in [
    (target_adf, "target", "k", "--"),
    (adf_before, "perturbed", "#cf4040", "-"),
    (adf_after, "optimized", "#2d6a4f", "-"),
]:
    adf_v = data[0].detach().numpy()
    adf_v = adf_v / (adf_v.sum() * dphi) if adf_v.sum() > 0 else adf_v
    axs[2].plot(phi_guide, adf_v, lw=2, ls=ls, color=color, label=label)

axs[2].set_xlabel("angle (deg)")
axs[2].set_ylabel("P(phi)")
axs[2].set_title("ADF recovery")
axs[2].legend()

plt.tight_layout()
plt.show()

## Compare: optimized vs target in widget

In [ ]:
PDFADFWidget(calc_guide, g2_after, adf_after)

## Load from ASE

If you have an existing ASE Atoms object (e.g. from a CIF or xyz file), convert to torch tensors and compute.

In [ ]:
from ase.build import bulk

def atoms_to_torch(atoms, dtype=torch.float64):
    """Convert ASE Atoms to torch tensors for DifferentiablePDFADF."""
    pos = torch.tensor(atoms.positions, dtype=dtype, requires_grad=True)
    sp = torch.tensor(atoms.numbers, dtype=torch.long)
    cl = torch.tensor(atoms.cell.array, dtype=dtype)
    return pos, sp, cl

# example: bulk silicon from ASE
si_ase = bulk("Si", "diamond", a=5.43) * (4, 4, 4)
pos_ase, sp_ase, cell_ase = atoms_to_torch(si_ase)
print(f"ASE Si: {len(si_ase)} atoms, cell = {np.round(si_ase.cell.lengths(), 2)}")

species_list = sorted(set(si_ase.numbers.tolist()))
calc_ase = DifferentiablePDFADF(
    r_max=8.0, r_step=0.05, phi_num_bins=90,
    sigma_r=0.1, sigma_phi=0.05, species=species_list,
).double()

g2_ase, adf_ase = calc_ase.compute(pos_ase, sp_ase, cell_ase)
PDFADFWidget(calc_ase, g2_ase, adf_ase)